## Example 1: OTLP gRPC trace exporter

In [ ]:
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExportResult
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

print("Example 1 - OTLP gRPC trace exporter")

# Nothing listens on 127.0.0.1:9 (the discard port), so every export below must return
# FAILURE instead of raising or hanging; timeout=1 keeps the exporters' retry loop short.
UNREACHABLE = "127.0.0.1:9"


def record_spans(*names):
    """Finish one span per name with a local tracer provider and return the finished spans."""
    store = InMemorySpanExporter()
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(store))
    tracer = provider.get_tracer("example", "1.0")
    for name in names:
        with tracer.start_as_current_span(name) as span:
            span.set_attribute("example.name", name)
    return store.get_finished_spans()


def record_metrics(resource=None):
    """Record a counter with a local meter provider and return the collected MetricsData."""
    reader = InMemoryMetricReader()
    provider = MeterProvider(metric_readers=[reader], **({"resource": resource} if resource else {}))
    counter = provider.get_meter("example", version="1.0").create_counter("requests")
    counter.add(1, {"method": "GET"})
    return reader.get_metrics_data()


# Create gRPC OTLP exporter and send it a finished span
exporter = OTLPSpanExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)
print(f"Created: {type(exporter).__name__}")
print(f"Protocol: gRPC")

result = exporter.export(record_spans("grpc_span"))
print(f"export() to {UNREACHABLE}: {result}")
assert result is SpanExportResult.FAILURE, result
exporter.shutdown()

print("Example 1 completed successfully")

## Example 2: OTLP HTTP trace exporter

In [ ]:
# Reuses UNREACHABLE and record_spans() from Example 1.
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SpanExportResult

print("Example 2 - OTLP HTTP trace exporter")

# Create HTTP OTLP exporter
endpoint = f"http://{UNREACHABLE}/v1/traces"
exporter = OTLPSpanExporter(endpoint=endpoint, timeout=1)

print(f"Created: {type(exporter).__name__}")
print(f"Protocol: HTTP")
print(f"Endpoint: {endpoint}")

result = exporter.export(record_spans("http_span"))
print(f"export(): {result}")
assert result is SpanExportResult.FAILURE, result
exporter.shutdown()

print("Example 2 completed successfully")

## Example 3: OTLP metrics exporter

In [ ]:
# Reuses UNREACHABLE and record_metrics() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import MetricExportResult

print("Example 3 - OTLP metrics exporter")

# Create metrics exporter
exporter = OTLPMetricExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)
print(f"Created: {type(exporter).__name__}")

# Record a counter and export the collected metrics
metrics_data = record_metrics()
assert metrics_data.resource_metrics[0].scope_metrics[0].metrics[0].name == "requests"
result = exporter.export(metrics_data, timeout_millis=1000)
print(f"export(): {result}")
assert result is MetricExportResult.FAILURE, result
exporter.shutdown()

print("Example 3 completed successfully")

## Example 4: OTLP with headers

In [ ]:
# Reuses UNREACHABLE and record_spans() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SpanExportResult

print("Example 4 - OTLP with headers")

# Create exporter with authentication headers (placeholder values)
exporter = OTLPSpanExporter(
    endpoint=f"http://{UNREACHABLE}",
    insecure=True,
    timeout=1,
    headers=(
        ("authorization", "Bearer token123"),
        ("x-api-key", "key123")
    )
)

print(f"Exporter with auth headers configured")

result = exporter.export(record_spans("auth_span"))
print(f"export(): {result}")
assert result is SpanExportResult.FAILURE, result
exporter.shutdown()

print("Example 4 completed successfully")

## Example 5: Multiple OTLP exporters

In [ ]:
# Reuses UNREACHABLE, record_spans() and record_metrics() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter as GRPCSpanExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter as HTTPSpanExporter
from opentelemetry.sdk.metrics.export import MetricExportResult
from opentelemetry.sdk.trace.export import SpanExportResult

print("Example 5 - Multiple OTLP exporters")

# Create gRPC trace exporter
grpc_trace = GRPCSpanExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

# Create HTTP trace exporter
http_trace = HTTPSpanExporter(endpoint=f"http://{UNREACHABLE}/v1/traces", timeout=1)

# Create metrics exporter
metric_exporter = OTLPMetricExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

print(f"gRPC Trace: {type(grpc_trace).__name__}")
print(f"HTTP Trace: {type(http_trace).__name__}")
print(f"Metrics: {type(metric_exporter).__name__}")

# The same finished spans go to both trace exporters
spans = record_spans("fan_out_span_1", "fan_out_span_2")
for exporter in (grpc_trace, http_trace):
    result = exporter.export(spans)
    assert result is SpanExportResult.FAILURE, (type(exporter).__module__, result)
    exporter.shutdown()
result = metric_exporter.export(record_metrics(), timeout_millis=1000)
assert result is MetricExportResult.FAILURE, result
metric_exporter.shutdown()
print("All three exporters reported FAILURE for the unreachable collector")

print("Example 5 completed successfully")